**Role in the paper.** Computes the p2a method (*p2a_MS_dist*, after Michau et al. 2023) for one city
(called "method 2 / blended approach" in the file names):
- simulated urban signal: the grid cell nearest to the urban MS station among the cells with urban fraction above a threshold (50 % by default, 65 % for Barcelona and Berlin, 20 % for Turku; set in the urban-cell selection cell);
- rural reference: mean over a rural mask (within a radius of the centre, nature fraction >= 0.5,
  orography within a threshold of the city centre); the observed rural reference is CERRA averaged over the same mask
  (ANASTASIA for Rennes).

Output: `results/data/<City>/method_2_blended_approach/<City>_UHI_method2_{simulation,observations}.nc`, which
`method_3_urclimask/Eurocities_urclimmask_<City>_.ipynb` reads.

# UHI Blended approach (method 2)

In [ ]:
import numpy as np
import metobs_toolkit 
import os
from pathlib import Path
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import glob
import cartopy.crs as ccrs
import cartopy
import osmnx as ox
from rasterio import warp
import rasterio
import sys
import importlib
from pathlib import Path
import geopandas as gpd
ROOT = Path.cwd().parent.parent.parent

In [ ]:
src_dir = ROOT / '/scripts/analysis/method_2_blended_approach'

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import src.analysis_blended_approach as analysis_blended_approach
import src.load_data as load_data

importlib.reload(analysis_blended_approach)
importlib.reload(load_data)

from src.load_data import (
    load_simulation_data,
    prepare_simulation_for_uhi,
    load_observations,
    prepare_observation_inputs_for_uhi,
    observations_hourly_to_monthly_mean_daily,
    load_ucdb_city,
 )
from src.analysis_blended_approach import (
    add_weighted_timeseries,
    crop_area_city_radius,
    create_rural_mask,
    align_to_reference_grid,
    compute_spatial_mean_over_mask,
    from_latlon_to_xy
 )

## Set variables

In [ ]:
city = 'Berlin'
urban_station = 'GMN00000433'
lon_city = 13.405708562830844
lat_city = 52.520355269780985

network = 'meteo_stations'
subset = ''
start = "2020-01-01"
stop = "2021-12-31"

#Ghent (51.05454277446639, 3.7249005076743438)

domain = 'EUROCITIES'
domain_resolution = 2.5

radius = 50 # in km, defines the size of the area around the city center to consider for the analysis.

## Load required data

In [ ]:
data_dir =  ROOT / f'data/'
plot_dir =  ROOT / f'results/figures/{city}/method_2_blended_approach'
os.makedirs(plot_dir, exist_ok=True)
save_dir =  ROOT / f'results/data/{city}/method_2_blended_approach'
os.makedirs(save_dir, exist_ok=True)

In [ ]:
correction = 273.15 #convert from K to °C

# load monthly mean simulation data (preproccessed)

path_tasmin = f"{data_dir}/processed/simulations/monthly_means/{domain}_operational_cy43_tasmin_monmean_*.nc"

da_tasmin = xr.open_mfdataset(sorted(glob.glob(path_tasmin)), engine="netcdf4", chunks="auto").rename({'tas':'tasmin'}).sel(time=slice(start,stop)) 

#crop around city and merge into one dataset
da_tasmin = crop_area_city_radius(da_tasmin, radius_km=radius, lon_city=lon_city, lat_city=lat_city, margin_km=1)
da_crp = xr.merge([da_tasmin])

proj_dict = da_crp.crs.attrs
plot_proj = ccrs.LambertConformal(central_latitude=proj_dict["latitude_of_projection_origin"], central_longitude=proj_dict["longitude_of_central_meridian"], standard_parallels=proj_dict["standard_parallel"], globe=ccrs.Globe(semimajor_axis=proj_dict["earth_radius"], semiminor_axis=proj_dict["earth_radius"]))
plot_proj

da_crp

In [ ]:
#load static input data of simulation
static_input_path = f'{data_dir}/metadata/simulations/static_variables_eurocities.nc'
static_input = xr.open_dataset(static_input_path,engine='netcdf4')

#crop around city
static_input = crop_area_city_radius(static_input, radius_km=radius, lon_city=lon_city, lat_city=lat_city, margin_km=1)

# Align static information to model data coordinates (in case their is a rounding error in the grid coordinates causing a mismatch in the grid)
static_input, alignment_status = align_to_reference_grid(static_input, da_crp, coord_names=['x', 'y'])
print(alignment_status)

In [ ]:
#load reanalysis data (preproccessed: regridded to model grid and monthly mean of daily data)

cerra_path_tasmin = f"{data_dir}/processed/reanalysis/CERRA/CERRA_tmin_monmean_*_wb_regrid_EUROCITIES.nc"
cerra_tasmin = xr.open_mfdataset(sorted(glob.glob(cerra_path_tasmin)), engine="netcdf4", chunks="auto").rename({'valid_time':'time', 't2m':'tasmin'}).sel(time=slice(start,stop))

cerra_tasmin_crp = crop_area_city_radius(cerra_tasmin, radius_km=radius, lon_city=lon_city, lat_city=lat_city, margin_km=1)

cerra_crp = xr.merge([cerra_tasmin_crp])
# Align static information to model data coordinates
cerra_crp, alignment_status = align_to_reference_grid(cerra_crp, da_crp, coord_names=['x', 'y'])
print(alignment_status)

In [ ]:
# load station observation data (preproccessed: monthly mean of daily data)

obs_path_tasmin = f"{data_dir}/processed/observations/{city}/{network}/datasets/{city}_{network}_monthlymean_dailymin.nc"

da_obs_tasmin = xr.open_dataset(obs_path_tasmin, engine="netcdf4", chunks="auto").sel(time=slice(start,stop))

pd_meta = pd.read_csv(f"{data_dir}/processed/observations/{city}/{network}/{city}_{network}_metadata.csv", index_col='name')

lat_station, lon_station = pd_meta.loc[urban_station]['lat'], pd_meta.loc[urban_station]['lon']
x_station, y_station = from_latlon_to_xy(lat_station, lon_station, plot_proj)
print(lat_station, lon_station)

## Create rural mask

In [ ]:
rural_mask = create_rural_mask(static_input, lon_city=lon_city, lat_city=lat_city, buffer_radius_km=radius, nature_threshold=50, orography_threshold=750)


In [ ]:
ucdb_city = load_ucdb_city(f'{data_dir}/metadata/UCDB_shapefiles', city)

lon_min = float(static_input['lon'].min())
lon_max = float(static_input['lon'].max())
lat_min = float(static_input['lat'].min())
lat_max = float(static_input['lat'].max())

fig, axes = plt.subplots(2, 2, figsize=(14, 12), subplot_kw={'projection': ccrs.PlateCarree()})
axes = axes.ravel()

plot_specs = [
    (static_input['orog'], 'Orography (m)', 'terrain', None, None),
    (static_input['sfturf'], 'Urban fraction', 'Reds', None, None),
    (static_input['nature_fraction'], 'Nature fraction', 'Greens', None, None),
    (rural_mask['rural_mask'], 'Final mask', 'gray_r', 0, 1),
]

for ax, (data, title, cmap, vmin, vmax) in zip(axes, plot_specs):
    mesh = ax.pcolormesh(
        static_input['lon'],
        static_input['lat'],
        data,
        shading='auto',
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        transform=ccrs.PlateCarree(),
    )
    ax.add_geometries(
        ucdb_city.geometry,
        crs=ccrs.PlateCarree(),
        facecolor='none',
        edgecolor='red',
        linewidth=2.0,
    )
    ax.plot(lon_city, lat_city, marker='x', color='red', markersize=8, transform=ccrs.PlateCarree())
    ax.set_title(title)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    plt.colorbar(mesh, ax=ax, shrink=0.82, pad=0.02)

fig.suptitle(f'{city} rural mask definition blended', fontsize=16)
fig.tight_layout()
fig.savefig(f'{plot_dir}/{city}_ruralmask_method2.png', dpi=200, bbox_inches='tight')
plt.show()

## UHI analysis 

In [ ]:
import numpy as np

# Select urban cells
urban_cells = static_input.where(static_input["sfturf"] > 65)

# Stack x/y grid into a single dimension
urban_cells = urban_cells.stack(gridcell=("y", "x"))

# Remove non-urban cells
urban_cells = urban_cells.dropna("gridcell", subset=["sfturf"])

# Calculate distance to station
dist = np.sqrt(
    (urban_cells["x"] - x_station)**2 +
    (urban_cells["y"] - y_station)**2
)

# Find closest urban cell
idx = dist.argmin()

# Extract its coordinates
x_urban = urban_cells["x"].isel(gridcell=idx).item()
y_urban = urban_cells["y"].isel(gridcell=idx).item()

print(x_urban, y_urban)

# Select the corresponding time series from da_crp
da_crp_urban = da_crp.sel(
    x=x_urban,
    y=y_urban,
    method="nearest"
)

In [ ]:
#make a plot of the mask , the urban fraction and the location of the selected point in da_crp_urban, the city centre and the ariport location plot all on xy grid use plot proj 
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': plot_proj})
mesh = ax.pcolormesh(
    static_input['x'],
    static_input['y'],
    static_input['sfturf'],
    shading='auto',
    cmap='Reds',
    transform=plot_proj,
)
#plot rural mask as hatched area over the urban fraction
rural_da = rural_mask['rural_mask']
rural_da = rural_da.where(rural_da == 1) # keep only the rural areas
rural_da = rural_da.fillna(0) # set the non-rural areas to 0
ax.contourf(
    static_input['x'],
    static_input['y'],
    rural_da,
    levels=[0.5, 1],
    colors='limegreen',
    hatches=['++', None],
    transform=plot_proj,
    alpha=0,    label='Rural mask'
)

x_city, y_city = from_latlon_to_xy(lat_city, lon_city, plot_proj)
ax.plot(x_city, y_city, marker='x', color='yellow', markersize=10, transform=plot_proj, label='City center')
ax.plot(x_station, y_station, marker='x', color='magenta', markersize=10, transform=plot_proj, label='Airport (urban ref obs)')
ax.plot(da_crp_urban['x'], da_crp_urban['y'], marker='x', color='lime', markersize=10, transform=plot_proj, label='closest point to airport with urban fraction > 50% (urban ref sim)')
#plot ucdb boundary
ucdb_city = ucdb_city = load_ucdb_city(f'{data_dir}/metadata/UCDB_shapefiles', city)
ax.add_geometries(
    ucdb_city.geometry,
    crs=ccrs.PlateCarree(),
    facecolor='none',
    edgecolor='red',
    linewidth=2.0,
    label='UCDB boundary'
)

ax.set_title(f'{city} - Urban fraction and selected point')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.legend()

plt.colorbar(mesh, ax=ax, shrink=0.5, pad=0.02)
plt.grid()
plt.savefig(f'{plot_dir}/{city}_urban_fraction_and_selected_point.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Compute spatially-averaged tas within the rural mask (time series)
# Using compute_spatial_mean_over_mask function from UHI_metric module
mask_da = rural_mask['rural_mask']

# Apply the function to compute spatial mean over rural_mask for all variables in tas_crp
sim_ts = compute_spatial_mean_over_mask(
    dataset=da_crp,
    mask_da=mask_da,
    variables=['tasmin']  # None = use all data variables (tas in this case)
)
sim_ts = sim_ts.rename({'tasmin':'tasmin_sim_rur'})

sim_ts['tasmin_urb'] = da_crp_urban['tasmin']

#calculate the UHI defined as the difference between the urban point and the rural modelled temperature
sim_ts['UHI_tasmin'] = sim_ts['tasmin_urb'] - sim_ts['tasmin_sim_rur']



In [ ]:
# Compute spatially-averaged tas within the rural mask (time series)
# Using compute_spatial_mean_over_mask function from UHI_metric module
mask_da = rural_mask['rural_mask']

# Apply the function to compute spatial mean over rural_mask for all variables in tas_crp
obs_ts = compute_spatial_mean_over_mask(
    dataset=cerra_crp,
    mask_da=mask_da,
    variables=['tasmin']  # None = use all data variables (tas in this case)
)

# Show a quick summary
obs_ts= obs_ts.rename({'tasmin':'tasmin_cerra_rur'})


#assign the monthly mean time series to new varialbes in the spatial_mean_ts_ana dataset
obs_ts['tasmin_stat_urb'] = (('time'),da_obs_tasmin[urban_station].values)

#calculate UHI defined as the difference between the urban station and the rural modelled temperature
obs_ts['UHI_tasmin'] = obs_ts['tasmin_stat_urb'] - obs_ts['tasmin_cerra_rur']


In [ ]:
obs_ts.UHI_tasmin.values

In [ ]:
#plot annual cycle of UHI_tasmin (monthly mean of daily UHI_tasmin)
uhi_tasmin_monmean_obs = obs_ts['UHI_tasmin'].groupby('time.month').mean()

uhi_tasmin_monmean_sim = sim_ts['UHI_tasmin'].groupby('time.month').mean()

plt.figure(figsize=(10,6))
uhi_tasmin_monmean_obs.plot(label='UHI (observation)')
uhi_tasmin_monmean_sim.plot(label='UHI (simulation)')

plt.title(f'{city} \n Annual cycle of UHI using blended approach (urban point vs rural area)')
plt.xlabel('months')
#change x axis ticks to month names
plt.xticks(ticks=np.arange(1,13), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.ylabel('UHI (°C)')
plt.ylim(-1, 3)
plt.legend()
plt.grid()
plt.savefig(f'{plot_dir}/{city}_UHI_annual_cycle_method2_tasmin.png', dpi=200, bbox_inches='tight')
plt.show()

## Save the p2a UHI time series
Read by `method_3_urclimask/Eurocities_urclimmask_<City>_.ipynb`.

In [ ]:
sim_ts.to_netcdf(f'{save_dir}/{city}_UHI_method2_simulation.nc')
obs_ts.to_netcdf(f'{save_dir}/{city}_UHI_method2_observations.nc')